# B3 ONLY — locked continuation

B3-nodiff is excluded. Imported historical B3 records retain their original provenance.


In [ ]:
from pathlib import Path
import sys, os, json, subprocess, traceback, shutil, platform, importlib.metadata, zipfile, hashlib, time, re
from datetime import datetime, timezone

SESSION_STARTED = time.time()
DIAG = Path('/kaggle/working/grace_diagnostics')
DIAG.mkdir(parents=True, exist_ok=True)
EVENTS = DIAG / 'events.log'

def note(message):
    text = str(message)
    print(text, flush=True)
    with EVENTS.open('a', encoding='utf-8') as f:
        f.write(f'[{datetime.now(timezone.utc).isoformat()}] {text}\n')

def diagnostics_zip():
    archive = shutil.make_archive('/kaggle/working/grace_diagnostics', 'zip', root_dir=DIAG)
    print('Tải log chẩn đoán từ Kaggle Output:', archive, flush=True)
    return archive

def run_logged(cmd, name, env=None):
    log = DIAG / f'{name}.log'
    note(f'Bắt đầu {name}: {cmd!r} | log đầy đủ: {log}')
    is_inference = name.startswith('resume_seed_')
    with log.open('w', encoding='utf-8') as f:
        proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                                text=True, encoding='utf-8', errors='replace', bufsize=1, env=env)
        for line in proc.stdout:
            f.write(line); f.flush()
            if re.search(r'\b(?:Found|Skip(?:ping)?) link\b', line, flags=re.IGNORECASE):
                continue
            if is_inference:
                match = re.search(r'^(\S+).*\((\d+)/(\d+)\)\s*$', line.strip())
                if match:
                    run_key, done, total = match.group(1), int(match.group(2)), int(match.group(3))
                    if done % 100 == 0 or done == total:
                        print(f'Inference {run_key}: {done}/{total}', flush=True)
                elif line.strip():
                    print(line, end='', flush=True)
            else:
                print(line, end='', flush=True)
        code = proc.wait()
    if is_inference and '--status' in cmd:
        status_path = Path(cmd[cmd.index('--status') + 1])
        if status_path.is_file():
            status = json.loads(status_path.read_text(encoding='utf-8'))
            print(f"Inference summary: {status.get('completed_records_total', 0)} records; state={status.get('state')}", flush=True)
            for run_key, progress in status.get('runs', {}).items():
                print(f"  {run_key}: {progress.get('completed', 0)}/{progress.get('count', '?')} [{progress.get('state', 'pending')}]")
    note(f'{name} kết thúc với mã {code}')
    if code:
        raise subprocess.CalledProcessError(code, cmd)

def versions():
    result = {}
    for name in ('llama-cpp-python', 'transformers', 'huggingface_hub', 'scikit-learn', 'numpy', 'scipy'):
        try: result[name] = importlib.metadata.version(name)
        except importlib.metadata.PackageNotFoundError: result[name] = 'not installed'
    note('Installed package versions: ' + json.dumps(result))
    return result

def capture_failure(shell, etype, evalue, tb, tb_offset=None):
    trace = ''.join(traceback.format_exception(etype, evalue, tb))
    try:
        with (DIAG / 'exceptions.log').open('a', encoding='utf-8') as f:
            f.write(f'\n[{datetime.now(timezone.utc).isoformat()}]\n{trace}\n')
        note(f'Lỗi: {etype.__name__}: {evalue}')
        versions()
        diagnostics_zip()
    except Exception as diagnostic_error:
        print('Không thể ghi đầy đủ chẩn đoán:', diagnostic_error, flush=True)
    shell.showtraceback((etype, evalue, tb), tb_offset=tb_offset)
    return []

get_ipython().set_custom_exc((Exception,), capture_failure)
note('UTC: ' + datetime.now(timezone.utc).isoformat())
note('Python: ' + sys.version)
note('Executable: ' + sys.executable)
note('Platform: ' + platform.platform())
for loc in ('/kaggle/working', '/kaggle/input'):
    try:
        usage = shutil.disk_usage(loc)
        note(f'Disk {loc}: total={usage.total} free={usage.free} bytes')
    except OSError as exc: note(f'Disk {loc}: {exc}')
versions()
for name, cmd in [('gpu', ['nvidia-smi']), ('cuda_compiler', ['nvcc', '--version']), ('cmake', ['cmake', '--version']), ('pip', [sys.executable, '-m', 'pip', '--version'])]:
    try: run_logged(cmd, name)
    except (OSError, subprocess.CalledProcessError) as exc: note(f'{name}: {type(exc).__name__}: {exc}')
try:
    input_entries = [p.name for p in Path('/kaggle/input').glob('*')]
    note(f'Input entries: {input_entries}')
    gguf_files = [str(p) for p in Path('/kaggle/input').rglob('*.gguf')]
    note(f'Input GGUF files: {gguf_files}')
except Exception as exc: note(f'Input inspection error: {exc}')
diagnostics_zip()
RUNTIME_LOCK_BYTES = b'{\n  "schema_version": 1,\n  "contract_id": "grace_qwen_q4_runtime_v1",\n  "created_on": "2026-11-07",\n  "timezone": "Asia/Bangkok",\n  "status": "core_policy_locked_notebook_adoption_pending",\n  "scope": "New GRACE diagnostic Qwen Q4 inference on Kaggle only; excludes Colab/Vast and preparation/CodeT5/legacy API prototypes",\n  "user_decisions": {\n    "inference_seed": 4294967295,\n    "packages": "exact versions approved",\n    "gpu_policy": "layer + auto free VRAM, approved",\n    "cpu_threads": "approved: preserve default CPU-count formulas used by all 16 notebooks",\n    "build": "approved: Kaggle platform build with mandatory complete provenance; no shared container required"\n  },\n  "model": {\n    "repo": "ggml-org/Qwen3-Coder-30B-A3B-Instruct-GGUF",\n    "filename": "Qwen3-Coder-30B-A3B-Instruct-Q4_K_M.gguf",\n    "sha256": "f6050b4da22869aa3fb9d94bb3314e9fc5a027d4741c1f97761c387d6b2b2335",\n    "quantization": "Q4_K_M",\n    "verify_sha256_before_load": true,\n    "adapters_allowed": false\n  },\n  "tokenizer": {\n    "repo": "Qwen/Qwen3-Coder-30B-A3B-Instruct",\n    "revision": "b2cff646eb4bb1d68355c01b18ae02e7cf42d120",\n    "chat_template": {\n      "role": "user",\n      "add_generation_prompt": true\n    },\n    "gguf_tokenize": {\n      "add_bos": false,\n      "special": true\n    },\n    "require_hf_gguf_token_id_equality": true,\n    "require_single_token_binary_labels": true\n  },\n  "packages": {\n    "numpy": "2.1.3",\n    "huggingface-hub": "0.36.2",\n    "tokenizers": "0.22.2",\n    "transformers": "4.57.1",\n    "llama-cpp-python": "0.3.16",\n    "scikit-learn": "1.5.2"\n  },\n  "backend_source": {\n    "name": "llama-cpp-python",\n    "version": "0.3.16",\n    "sdist_filename": "llama_cpp_python-0.3.16.tar.gz",\n    "sdist_sha256": "34ed0f9bd9431af045bb63d9324ae620ad0536653740e9bb163a2e1fcb973be6",\n    "verify_sdist_before_build": true,\n    "allow_other_backend": false,\n    "record_runner_source_sha256": true\n  },\n  "constructor": {\n    "n_ctx": 8192,\n    "n_batch": 256,\n    "n_ubatch": 256,\n    "n_gpu_layers": -1,\n    "split_mode": "LLAMA_SPLIT_MODE_LAYER",\n    "main_gpu": 0,\n    "tensor_split": null,\n    "seed": 4294967295,\n    "logits_all": true,\n    "flash_attn": false,\n    "offload_kqv": true,\n    "use_mmap": true,\n    "use_mlock": false,\n    "embedding": false\n  },\n  "gpu": {\n    "profile": "kaggle_t4_x2",\n    "device_count": 2,\n    "device_model": "NVIDIA Tesla T4",\n    "distribution": "layer_by_free_vram_at_load",\n    "random_distribution": false,\n    "fixed_ratio_guaranteed": false,\n    "record_free_and_total_vram": true,\n    "record_device_order_and_uuid": true,\n    "record_layer_and_buffer_offload_report": true,\n    "required_offloaded_layers": 49,\n    "allow_silent_cpu_layer_fallback": false,\n    "allow_auxiliary_host_buffers": true,\n    "other_hardware_requires_explicit_profile_review": true\n  },\n  "threads": {\n    "policy": "preserve_backend_default_formulas",\n    "n_threads": "max(multiprocessing.cpu_count() // 2, 1)",\n    "n_threads_batch": "multiprocessing.cpu_count()",\n    "fixed_thread_count": null,\n    "record_effective_counts": true,\n    "approval_status": "user_approved"\n  },\n  "build": {\n    "GGML_CUDA": true,\n    "CMAKE_CUDA_ARCHITECTURES": "75",\n    "CMAKE_LIBRARY_PATH": "/usr/local/nvidia/lib64",\n    "CMAKE_BUILD_PARALLEL_LEVEL": "2",\n    "FORCE_CMAKE": "1",\n    "python_family": "3.13",\n    "exact_python_patch": null,\n    "exact_cuda_toolkit": null,\n    "exact_host_compiler": null,\n    "exact_cmake": null,\n    "compiled_wheel_sha256": null,\n    "binary_environment_fully_locked": false,\n    "policy": "Kaggle platform build is approved. Record exact Python patch, CUDA/nvcc/driver, compiler, CMake, flags, dependencies and compiled backend hash. A shared container or bit-identical binary is not required or claimed."\n  },\n  "scoring": {\n    "name": "unconstrained_first_token_logits_v1",\n    "reset_before_each_prompt": true,\n    "evaluate": "llm.eval(gguf_ids)",\n    "logits": "llm.scores[llm.n_tokens - 1]",\n    "normalization_dtype": "float64",\n    "logprob": "stable full-vocabulary log_softmax of last-position logits",\n    "s": "exp(logprob_1 - logaddexp(logprob_0, logprob_1))",\n    "pred_argmax": "int(s >= 0.5)",\n    "raw_token": "argmax of full-vocabulary logits",\n    "reject_non_finite": true,\n    "retain_valid_score_if_raw_not_binary": true\n  },\n  "decoding": {\n    "mode": "direct_logits_and_greedy_argmax",\n    "sampling": false,\n    "generation_api": null,\n    "temperature": null,\n    "top_p": null,\n    "top_k": null,\n    "note": "Sampling parameters are not used by eval; the default-seed sentinel does not promise fixed RNG output for future sampling paths."\n  },\n  "prompt_contract": {\n    "prompt_selection_seeds": [\n      42,\n      43,\n      44\n    ],\n    "selection_seed_distinct_from_inference_seed": true,\n    "identity": [\n      "arm",\n      "prompt seed",\n      "dataset",\n      "split",\n      "ID",\n      "bundle SHA256",\n      "prompt SHA256"\n    ],\n    "verify_frozen_input_and_each_prompt_hash": true,\n    "silent_reformatting_or_truncation": false,\n    "different_arms_use_their_own_frozen_prompts": true\n  },\n  "thresholds": {\n    "source": "existing frozen arm-specific threshold contracts",\n    "test_refit_allowed": false,\n    "preserve_existing_probe_inheritance": true\n  },\n  "resume": {\n    "require_match": [\n      "arm",\n      "prompt seed",\n      "dataset",\n      "split",\n      "ID",\n      "label",\n      "prompt SHA256",\n      "model SHA256",\n      "tokenizer revision",\n      "backend source",\n      "scoring",\n      "constructor",\n      "threshold lock when applicable"\n    ],\n    "changed_contract": "do not silently append to historical results; use separate run identity or reviewed migration",\n    "rewrite_historical_outputs": false\n  },\n  "provenance_required": [\n    "contract SHA256",\n    "all package versions including transitive dependencies",\n    "exact Python/OS/libc",\n    "CUDA toolkit/nvcc/driver",\n    "GPU device order/UUID/VRAM",\n    "host compiler/CMake/build flags",\n    "source archive SHA256",\n    "compiled backend/wheel SHA256",\n    "runner source SHA256",\n    "effective CPU threads",\n    "effective batch/microbatch",\n    "offload log",\n    "model/tokenizer/prompt/threshold identities"\n  ],\n  "integration": {\n    "existing_notebooks_auto_enforce_json": false,\n    "required_before_claiming_compliance": "load, validate and apply this JSON; abort on incompatible values",\n    "historical_runs_are_not_retroactively_certified": true\n  },\n  "open_questions": [\n    {\n      "id": "missing_run_logs",\n      "arms": [\n        "B1-full",\n        "B1-shuf",\n        "B2-rep",\n        "B2-nolabel",\n        "B2-labelonly"\n      ],\n      "status": "awaiting executed notebooks/install logs"\n    },\n    {\n      "id": "b3_runners",\n      "arms": [\n        "B3",\n        "B3-nodiff",\n        "B3-swap",\n        "B3-flip"\n      ],\n      "status": "no dedicated current Kaggle Qwen notebook found in GRACE-summary"\n    }\n  ],\n  "references": {\n    "package_hash": "https://pypi.org/pypi/llama-cpp-python/0.3.16/json",\n    "constructor_defaults": "https://raw.githubusercontent.com/abetlen/llama-cpp-python/v0.3.16/llama_cpp/llama.py",\n    "auto_split": "https://raw.githubusercontent.com/ggml-org/llama.cpp/4227c9be4268ac844921b90f31595f81236bd317/src/llama-model.cpp"\n  },\n  "notebook_audit": [\n    {\n      "path": "01_Author_Baseline_Unified/notebooks/02_kaggle_qwen_q4_b2.ipynb",\n      "file_sha256_at_audit": "d83c14d4f069e1987c024b27ac801840f7fa080d6fd3f75a17f2e004fdce9ba3",\n      "configured_inference_seed": 4294967295,\n      "tensor_split": null,\n      "missing_explicit_package_pins": [\n        "numpy",\n        "huggingface-hub",\n        "tokenizers"\n      ],\n      "shared_json_integrated": false\n    },\n    {\n      "path": "02_Arm_B0_Source_Only/notebooks/02_kaggle_qwen_q4_b0.ipynb",\n      "file_sha256_at_audit": "fdc0ee82e6df6903463023471e2babef6fd6a136bfc4ce14e7dd8806376ee7d6",\n      "configured_inference_seed": 42,\n      "tensor_split": null,\n      "missing_explicit_package_pins": [\n        "numpy",\n        "huggingface-hub",\n        "tokenizers"\n      ],\n      "shared_json_integrated": false\n    },\n    {\n      "path": "03_Arm_B1_Graph_Only/notebooks/02_kaggle_qwen_q4_b1.ipynb",\n      "file_sha256_at_audit": "c01dac2696b9391414d0f9c5ee0f7eb1f26e1ef0f69de13b953da83f75f18746",\n      "configured_inference_seed": 42,\n      "tensor_split": null,\n      "missing_explicit_package_pins": [\n        "numpy",\n        "huggingface-hub",\n        "tokenizers"\n      ],\n      "shared_json_integrated": false\n    },\n    {\n      "path": "03_Arm_B1_Graph_Only/notebooks/02_kaggle_qwen_q4_b1_executed_I08.ipynb",\n      "file_sha256_at_audit": "6e501ab2f57b964e9c43a807297df4407620bcb9a02f0a3744d742341dce22fb",\n      "configured_inference_seed": 42,\n      "tensor_split": null,\n      "missing_explicit_package_pins": [\n        "numpy",\n        "huggingface-hub",\n        "tokenizers"\n      ],\n      "shared_json_integrated": false\n    },\n    {\n      "path": "04_Arm_B2_Flip/notebooks/02_kaggle_qwen_q4_b2_flip.ipynb",\n      "file_sha256_at_audit": "c6e7ce9c1351ab88375cd6e4d9b7d34cd5bb9f7862108fef5d12d81196cf37fb",\n      "configured_inference_seed": 42,\n      "tensor_split": null,\n      "missing_explicit_package_pins": [\n        "numpy",\n        "huggingface-hub",\n        "tokenizers"\n      ],\n      "shared_json_integrated": false\n    },\n    {\n      "path": "06_Arm_B2_Rand/notebooks/02_kaggle_qwen_q4_b2_rand.ipynb",\n      "file_sha256_at_audit": "b819048b5f126882dba6ff59581354937101195d8807974bc3a1f7cc65910c86",\n      "configured_inference_seed": 4294967295,\n      "tensor_split": null,\n      "missing_explicit_package_pins": [],\n      "shared_json_integrated": false\n    },\n    {\n      "path": "06_Arm_B2_Rand/test_delivery_20261107/02_kaggle_qwen_q4_b2_rand_seed42_test.ipynb",\n      "file_sha256_at_audit": "a35b30da7d32ca935f0d5bba894ccb4fa02ba52e1212c70d6afded2ba08e3c7e",\n      "configured_inference_seed": 4294967295,\n      "tensor_split": null,\n      "missing_explicit_package_pins": [],\n      "shared_json_integrated": false\n    },\n    {\n      "path": "06_Arm_B2_Rand/test_delivery_20261107/02_kaggle_qwen_q4_b2_rand_seed43_test.ipynb",\n      "file_sha256_at_audit": "42ebe992c4e716ced593efdd089a720d3d0019663b69f5faf303c201975264d1",\n      "configured_inference_seed": 4294967295,\n      "tensor_split": null,\n      "missing_explicit_package_pins": [],\n      "shared_json_integrated": false\n    },\n    {\n      "path": "11_Arm_B1H2_Evidence/kaggle_b1h2_test_20261106/02_kaggle_qwen_q4_b1_h2_test_20261106.ipynb",\n      "file_sha256_at_audit": "3cb1f4d4fc2d4e9ca839d22db2b1660f671aebbd046bdd2a0e2e94f4a8541dc7",\n      "configured_inference_seed": 4294967295,\n      "tensor_split": [\n        1.0,\n        1.0\n      ],\n      "missing_explicit_package_pins": [],\n      "shared_json_integrated": false\n    },\n    {\n      "path": "11_Arm_B1H2_Evidence/notebooks/02_kaggle_qwen_q4_b1_h2_validation_20261106.ipynb",\n      "file_sha256_at_audit": "57cdf5249395aec0f0c5bc47421a9d6fb3789fd5809de7d42bc80f632d91c88f",\n      "configured_inference_seed": 4294967295,\n      "tensor_split": [\n        1.0,\n        1.0\n      ],\n      "missing_explicit_package_pins": [\n        "numpy",\n        "huggingface-hub",\n        "tokenizers"\n      ],\n      "shared_json_integrated": false\n    },\n    {\n      "path": "Runs-management/kaggle_validation_6arms_20261104/02_kaggle_qwen_q4_b1_full_validation.ipynb",\n      "file_sha256_at_audit": "dcfff73568daddd1e97934c0f2175d4b36b36339032049fffb16160a541b34bf",\n      "configured_inference_seed": 4294967295,\n      "tensor_split": null,\n      "missing_explicit_package_pins": [\n        "numpy",\n        "huggingface-hub",\n        "tokenizers"\n      ],\n      "shared_json_integrated": false\n    },\n    {\n      "path": "Runs-management/kaggle_validation_6arms_20261104/02_kaggle_qwen_q4_b1_shuf_validation.ipynb",\n      "file_sha256_at_audit": "d9ed03125e8f309e28b5f06d14be3038e3fe77277fe9ae0ac33857d777aa9759",\n      "configured_inference_seed": 4294967295,\n      "tensor_split": null,\n      "missing_explicit_package_pins": [\n        "numpy",\n        "huggingface-hub",\n        "tokenizers"\n      ],\n      "shared_json_integrated": false\n    },\n    {\n      "path": "Runs-management/kaggle_validation_6arms_20261104/02_kaggle_qwen_q4_b2_labelonly_validation.ipynb",\n      "file_sha256_at_audit": "6a54f85e4537e66ab1a1c57615232e3b8a99850a70308c3d1bcadb0fcdb3d8db",\n      "configured_inference_seed": 4294967295,\n      "tensor_split": null,\n      "missing_explicit_package_pins": [\n        "numpy",\n        "huggingface-hub",\n        "tokenizers"\n      ],\n      "shared_json_integrated": false\n    },\n    {\n      "path": "Runs-management/kaggle_validation_6arms_20261104/02_kaggle_qwen_q4_b2_nolabel_validation.ipynb",\n      "file_sha256_at_audit": "099325d9a24e2df72e43a33f77250ea1590302409d7d5cb1b5a7a035f605364a",\n      "configured_inference_seed": 4294967295,\n      "tensor_split": null,\n      "missing_explicit_package_pins": [\n        "numpy",\n        "huggingface-hub",\n        "tokenizers"\n      ],\n      "shared_json_integrated": false\n    },\n    {\n      "path": "Runs-management/kaggle_validation_6arms_20261104/02_kaggle_qwen_q4_b2_rand.ipynb",\n      "file_sha256_at_audit": "ccef18aeb640a0ded8d5050e54e58a6e504cd0986460288425fba7a5c7dfed28",\n      "configured_inference_seed": 4294967295,\n      "tensor_split": null,\n      "missing_explicit_package_pins": [],\n      "shared_json_integrated": false\n    },\n    {\n      "path": "Runs-management/kaggle_validation_6arms_20261104/02_kaggle_qwen_q4_b2_rep_validation.ipynb",\n      "file_sha256_at_audit": "2e6bfb53c9fc4525a014a9d79ad862b5c3fbc40fd3bcf8e24729ab91257e7a8e",\n      "configured_inference_seed": 4294967295,\n      "tensor_split": null,\n      "missing_explicit_package_pins": [\n        "numpy",\n        "huggingface-hub",\n        "tokenizers"\n      ],\n      "shared_json_integrated": false\n    }\n  ],\n  "resolved_decisions": [\n    {\n      "id": "build_artifact",\n      "question": "Pin a shared container/build artifact or allow platform builds with provenance? All notebook builders leave exact CUDA/compiler/CMake/Python patch unpinned.",\n      "status": "resolved",\n      "decision": "Kaggle platform build with mandatory full provenance"\n    },\n    {\n      "id": "cpu_threads",\n      "question": "Keep CPU-count default formulas or lock absolute thread counts? All 16 constructors omit n_threads and n_threads_batch.",\n      "status": "resolved",\n      "interim_policy": "preserve existing formulas and record effective values",\n      "decision": "preserve default CPU-count formulas and record effective counts"\n    }\n  ],\n  "integrity": {\n    "algorithm": "sha256",\n    "canonicalization": "UTF-8 json.dumps(sort_keys=True,separators=(comma,colon),ensure_ascii=False); exclude integrity",\n    "sha256": "c0f17d3562541fa97e92b46413fa4eb56d2823c59cc9c1bf920a4c404ad3a171"\n  }\n}\n'
RUNTIME_LOCK_PATH = Path('/kaggle/working/inference_runtime_lock.json')
if RUNTIME_LOCK_PATH.exists() and RUNTIME_LOCK_PATH.read_bytes() != RUNTIME_LOCK_BYTES:
    raise ValueError('Working directory has a different runtime lock; start a fresh Kaggle session')
RUNTIME_LOCK_PATH.write_bytes(RUNTIME_LOCK_BYTES)
Path('/kaggle/working/grace_runtime_contract.py').write_text("import contextlib\nimport hashlib\nimport importlib.metadata\nimport json\nimport multiprocessing\nimport os\nimport platform\nimport re\nimport shutil\nimport subprocess\nimport sys\nfrom pathlib import Path\n\nEXPECTED_CONTRACT_SHA256 = '85d8770839f80260cdc7c27b27b6cce6bf8815675c8593db68fb102589c5cbd6'\nCONTRACT_PATH = Path('/kaggle/working/inference_runtime_lock.json')\n\ndef digest(path):\n    h = hashlib.sha256()\n    with Path(path).open('rb') as f:\n        for block in iter(lambda: f.read(1024 * 1024), b''): h.update(block)\n    return h.hexdigest()\n\ndef load_contract(path=CONTRACT_PATH):\n    raw = Path(path).read_bytes()\n    if hashlib.sha256(raw).hexdigest() != EXPECTED_CONTRACT_SHA256:\n        raise ValueError('Runtime contract file SHA-256 mismatch')\n    c = json.loads(raw)\n    unsigned = {k:v for k,v in c.items() if k != 'integrity'}\n    seal = hashlib.sha256(json.dumps(unsigned,sort_keys=True,separators=(',',':'),ensure_ascii=False).encode()).hexdigest()\n    if seal != c['integrity']['sha256']:\n        raise ValueError('Runtime contract canonical integrity mismatch')\n    return c\n\ndef command(args):\n    p = subprocess.run(args, check=True, text=True, capture_output=True)\n    return p.stdout + p.stderr\n\ndef assert_runtime(c):\n    actual = {name:importlib.metadata.version(name) for name in c['packages']}\n    if actual != c['packages']: raise RuntimeError(f'Package mismatch: {actual}')\n    if platform.system() != 'Linux' or '.'.join(map(str,sys.version_info[:2])) != c['build']['python_family']:\n        raise RuntimeError('Contract requires Kaggle Linux/Python 3.13')\n    gpu = command(['nvidia-smi','--query-gpu=index,uuid,name,memory.free,memory.total,driver_version','--format=csv,noheader'])\n    lines = [s for s in gpu.splitlines() if s.strip()]\n    if len(lines) != c['gpu']['device_count'] or any('T4' not in s for s in lines):\n        raise RuntimeError(f'Contract requires 2 T4 GPUs: {gpu}')\n    dist = importlib.metadata.distribution('llama-cpp-python')\n    origin = json.loads(dist.read_text('direct_url.json') or '{}')\n    if origin.get('archive_info',{}).get('hashes',{}).get('sha256') != c['backend_source']['sdist_sha256']:\n        raise RuntimeError('Backend installation is not from the verified source archive')\n    return actual, gpu\n\ndef provenance(c, runner, target):\n    packages, gpu = assert_runtime(c)\n    dist = importlib.metadata.distribution('llama-cpp-python')\n    binaries = {}\n    for entry in dist.files or []:\n        if str(entry).endswith(('.so','.dll','.dylib')):\n            p = Path(dist.locate_file(entry)); binaries[str(entry)] = digest(p)\n    if not binaries: raise RuntimeError('No compiled backend library found for provenance')\n    result = dict(runtime_contract_sha256=EXPECTED_CONTRACT_SHA256,\n        runtime_contract_integrity=c['integrity']['sha256'], packages=packages,\n        python=sys.version, platform=platform.platform(), libc=platform.libc_ver(),\n        gpu_before_load=gpu, cpu_count=multiprocessing.cpu_count(),\n        n_threads=max(multiprocessing.cpu_count()//2,1), n_threads_batch=multiprocessing.cpu_count(),\n        constructor=c['constructor'], runner_sha256=digest(runner),\n        cuda_device_order=os.environ.get('CUDA_DEVICE_ORDER'),\n        cuda_visible_devices=os.environ.get('CUDA_VISIBLE_DEVICES'),\n        backend_sdist_sha256=c['backend_source']['sdist_sha256'], compiled_backend_sha256=binaries,\n        build_environment={k:os.environ.get(k) for k in ('CMAKE_ARGS','CMAKE_BUILD_PARALLEL_LEVEL','FORCE_CMAKE')})\n    for name,args in [('nvcc',['nvcc','--version']),('compiler',['cc','--version']),\n                      ('cxx_compiler',['c++','--version']),('cmake',['cmake','--version']),\n                      ('pip_freeze',[sys.executable,'-m','pip','freeze','--all'])]:\n        result[name] = command(args)\n    target = Path(target); target.parent.mkdir(parents=True,exist_ok=True)\n    target.write_text(json.dumps(result,indent=2),encoding='utf-8')\n    return result\n\ndef load_locked_model(c, model_path, tokenizer_manifest, gpu_layers, log_path, model_digest):\n    import llama_cpp\n    if model_digest != c['model']['sha256']: raise ValueError('Frozen model SHA-256 mismatch')\n    if gpu_layers != c['constructor']['n_gpu_layers']: raise ValueError('GPU layers differ from lock')\n    if (tokenizer_manifest['tokenizer'],tokenizer_manifest['tokenizer_revision'],tokenizer_manifest['prompt_budget_tokens']) != (\n        c['tokenizer']['repo'],c['tokenizer']['revision'],c['constructor']['n_ctx']):\n        raise ValueError('Tokenizer/context differ from runtime lock')\n    if not llama_cpp.llama_supports_gpu_offload(): raise RuntimeError('CUDA backend unavailable')\n    kwargs = dict(c['constructor'])\n    kwargs['split_mode'] = getattr(llama_cpp,kwargs['split_mode'])\n    kwargs.update(n_threads=max(multiprocessing.cpu_count()//2,1), n_threads_batch=multiprocessing.cpu_count())\n    # The pinned Python binding's logger writes through sys.stderr, so capture that stream directly.\n    path = Path(log_path)\n    with path.open('w',encoding='utf-8') as log, contextlib.redirect_stderr(log):\n        model = llama_cpp.Llama(model_path=str(model_path),verbose=True,**kwargs)\n    text = path.read_text(encoding='utf-8',errors='replace')\n    print(text,flush=True)\n    reports = re.findall(r'offloaded\\s+(\\d+)\\s*/\\s*(\\d+)\\s+layers?\\s+to\\s+GPU',text,re.I)\n    required = c['gpu']['required_offloaded_layers']\n    if not any(int(a)==int(b)==required for a,b in reports):\n        model.close()\n        raise RuntimeError(f'Full {required}/{required} GPU offload not verified; inspect {path}')\n    for field in ('n_ctx', 'n_batch', 'n_ubatch'):\n        observed = re.findall(rf'\\b{field}\\s*=\\s*(\\d+)\\b', text)\n        if not observed or int(observed[-1]) != c['constructor'][field]:\n            model.close()\n            raise RuntimeError(f'Effective {field} differs from contract; inspect {path}')\n    # Both historical headers and the captured current backend format are accepted.\n    # Prefer devices actually used for model load when those lines are present.\n    loaded_ids = set(re.findall(\n        r'(?mi)^\\s*(?:[\\w.]+:\\s*)?using device CUDA(\\d+)\\s+\\([^\\n)]*\\bT4\\b[^\\n)]*\\)', text))\n    legacy_ids = set(re.findall(r'(?mi)^\\s*Device\\s+(\\d+):[^\\n]*\\bT4\\b', text))\n    observed_ids = loaded_ids if loaded_ids else legacy_ids\n    expected_ids = {str(i) for i in range(c['gpu']['device_count'])}\n    if observed_ids != expected_ids:\n        model.close()\n        raise RuntimeError(f'Expected T4 device IDs {expected_ids}, observed {observed_ids}; inspect {path}')\n    return model\n", encoding='utf-8')
if '/kaggle/working' not in sys.path:
    sys.path.insert(0, '/kaggle/working')
from grace_runtime_contract import assert_runtime, digest, load_contract, EXPECTED_CONTRACT_SHA256
RUNTIME = load_contract()
shutil.copy2(RUNTIME_LOCK_PATH, DIAG / RUNTIME_LOCK_PATH.name)


In [ ]:
KAGGLE_INPUT_ROOT = Path('/kaggle/input')
OUT_ROOT = Path('/kaggle/working/b3_only_locked_20261108')
BUNDLE_NAME = 'grace_locked_b3_only_migrated_20261108.zip'
MANIFEST_SHA = '40bfff9d68ae5212b1dc921fdedd06aed7daeed1000e6f832f08c915fd8b9780'
CHECKPOINT_NAME = 'b3_only_locked_20261108_checkpoint.zip'
MODEL_FILENAME = RUNTIME['model']['filename']
MAX_RUNTIME_HOURS = 9


In [ ]:
# Only a checkpoint with this run's exact identity can be restored.
checkpoints = list(KAGGLE_INPUT_ROOT.rglob(CHECKPOINT_NAME))
if len(checkpoints) > 1: raise ValueError('Ambiguous checkpoint inputs')
if checkpoints:
    with zipfile.ZipFile(checkpoints[0]) as z:
        identity = json.loads(z.read('run_identity.json'))
        if identity['input_manifest_sha256'] != MANIFEST_SHA or identity['runtime_contract_sha256'] != EXPECTED_CONTRACT_SHA256:
            raise ValueError('Checkpoint identity mismatch')
        for name in z.namelist():
            target = (OUT_ROOT / name).resolve()
            if not target.is_relative_to(OUT_ROOT.resolve()): raise ValueError('Unsafe checkpoint path')
        if z.testzip() is not None: raise ValueError('Checkpoint CRC failure')
        z.extractall(OUT_ROOT)
print('Approved new input manifest:', MANIFEST_SHA)

# Preserve hash-pinned evidence for the 29 imported runtime-locked source rows.
archives=list(KAGGLE_INPUT_ROOT.rglob(BUNDLE_NAME))
if len(archives)>1:raise ValueError('Ambiguous migration bundle')
if archives:
    migration_zip=zipfile.ZipFile(archives[0]);read_member=migration_zip.read
else:
    manifests=list(KAGGLE_INPUT_ROOT.rglob('input_manifest.json'))
    if len(manifests)!=1:raise ValueError('Ambiguous migration manifest')
    read_member=lambda name:(manifests[0].parent/name).read_bytes()
    migration_zip=None
try:
    raw_manifest=read_member('input_manifest.json')
    if hashlib.sha256(raw_manifest).hexdigest()!=MANIFEST_SHA:raise ValueError('Migration manifest mismatch')
    for ref in json.loads(raw_manifest)['migration_evidence']:
        payload=read_member(ref['member'])
        if hashlib.sha256(payload).hexdigest()!=ref['sha256']:raise ValueError('Migration evidence hash mismatch')
        target=(OUT_ROOT/ref['output_relative']).resolve()
        if not target.is_relative_to(OUT_ROOT.resolve()):raise ValueError('Unsafe migration output path')
        if target.exists() and target.read_bytes()!=payload:raise ValueError('Changed saved migration evidence')
        target.parent.mkdir(parents=True,exist_ok=True);target.write_bytes(payload)
finally:
    if migration_zip:migration_zip.close()


In [ ]:
# Install the approved versions, then build the verified source archive.
import urllib.request
build = RUNTIME['build']
os.environ['CMAKE_ARGS'] = (
    f"-DGGML_CUDA={'on' if build['GGML_CUDA'] else 'off'}"
    f" -DCMAKE_CUDA_ARCHITECTURES={build['CMAKE_CUDA_ARCHITECTURES']}"
    f" -DCMAKE_LIBRARY_PATH={build['CMAKE_LIBRARY_PATH']}"
)
for key in ('CMAKE_BUILD_PARALLEL_LEVEL', 'FORCE_CMAKE'):
    os.environ[key] = build[key]
packages = [f'{name}=={version}' for name, version in RUNTIME['packages'].items()
            if name != RUNTIME['backend_source']['name']]
packages += ['typing-extensions>=4.5.0', 'diskcache>=5.6.1', 'jinja2>=2.11.3']
run_logged([sys.executable, '-m', 'pip', 'install', '--no-cache-dir', *packages], 'install_dependencies')
source = RUNTIME['backend_source']
with urllib.request.urlopen(f"https://pypi.org/pypi/{source['name']}/{source['version']}/json", timeout=60) as response:
    metadata = json.load(response)
artifact = next(item for item in metadata['urls'] if item['filename'] == source['sdist_filename'])
if artifact['digests']['sha256'] != source['sdist_sha256']:
    raise ValueError('Backend source registry SHA-256 mismatch')
archive_path = Path('/kaggle/working') / source['sdist_filename']
with urllib.request.urlopen(artifact['url'], timeout=120) as response, archive_path.open('wb') as target:
    shutil.copyfileobj(response, target)
if digest(archive_path) != source['sdist_sha256']:
    raise ValueError('Backend source file SHA-256 mismatch')
run_logged([sys.executable, '-m', 'pip', 'install', '-v', '--no-cache-dir', '--no-deps',
            '--force-reinstall', str(archive_path)], 'install_llama_cuda')
note('Locked packages: ' + json.dumps(assert_runtime(RUNTIME)[0], sort_keys=True))
diagnostics_zip()


In [ ]:
assert_runtime(RUNTIME)
import llama_cpp
versions()
note('llama-cpp-python GPU offload: ' + str(llama_cpp.llama_supports_gpu_offload()))
assert llama_cpp.llama_supports_gpu_offload(), 'llama-cpp-python chưa có CUDA; xem install_llama_cuda.log'
diagnostics_zip()


In [ ]:
# Tìm đúng tên file model trong toàn bộ Input đã gắn, bất kể username/slug.
model_files = sorted(path for path in KAGGLE_INPUT_ROOT.rglob(MODEL_FILENAME) if path.is_file())
if len(model_files) != 1:
    raise FileNotFoundError(f'Cần đúng một {MODEL_FILENAME} trong Kaggle Input; '
                            f'tìm thấy: {list(map(str, model_files))}. '
                            'Hãy Add Input chứa model và bỏ bản trùng nếu có; '
                            'notebook không tự tải lại GGUF từ Hugging Face.')
model_path = str(model_files[0])
revision = 'local-input'
note(f'Dùng model input: {model_path}')
print('Model input:', model_path)
print('Batch output directory:', OUT_ROOT)


In [ ]:
Path('/kaggle/working/kaggle_inference.py').write_text('"""Kaggle worker: verified inputs, explicit legacy migration, locked new inference.\n\nThe notebook supplies the byte-verified grace_runtime_contract helper. Old rows\nremain unchanged and are explicitly excluded from new-runtime certification.\n"""\nimport argparse\nfrom datetime import datetime, timezone\nimport hashlib\nimport json\nimport math\nimport os\nfrom pathlib import Path\nimport shutil\nimport tempfile\nimport time\nfrom zipfile import ZIP_DEFLATED, ZipFile\n\nimport numpy as np\n\n\ndef digest_bytes(raw):\n    return hashlib.sha256(raw).hexdigest()\n\n\ndef canonical(row):\n    return digest_bytes(json.dumps(row, sort_keys=True, ensure_ascii=False, separators=(",", ":"), allow_nan=False).encode())\n\n\ndef jsonl(raw):\n    return [json.loads(line) for line in raw.decode("utf-8").splitlines() if line.strip()]\n\n\ndef atomic(path, value):\n    path = Path(path)\n    path.parent.mkdir(parents=True, exist_ok=True)\n    temporary = path.with_suffix(path.suffix + ".tmp")\n    with temporary.open("w", encoding="utf-8", newline="\\n") as f:\n        json.dump(value, f, ensure_ascii=False, indent=2, allow_nan=False)\n        f.write("\\n")\n        f.flush()\n        os.fsync(f.fileno())\n    os.replace(temporary, path)\n\n\ndef binary_scores(logits, label_ids):\n    values = np.asarray(logits, dtype=np.float64).reshape(-1)\n    if not np.all(np.isfinite(values)):\n        raise ValueError("Non-finite logits")\n    maximum = np.max(values)\n    normalizer = float(maximum + np.log(np.exp(values - maximum).sum()))\n    lp0 = float(values[label_ids[0]] - normalizer)\n    lp1 = float(values[label_ids[1]] - normalizer)\n    score = float(np.exp(lp1 - np.logaddexp(lp0, lp1)))\n    best = int(np.argmax(values))\n    top = np.argsort(values)[-5:][::-1]\n    return {"logprob_0": lp0, "logprob_1": lp1, "p0": math.exp(lp0), "p1": math.exp(lp1),\n            "s": score, "pred_argmax": int(score >= .5), "greedy_token_id": best,\n            "top5": [{"token_id": int(i), "logprob": float(values[i] - normalizer)} for i in top]}\n\n\ndef load_inputs(root, bundle_name, manifest_sha):\n    root = Path(root)\n    packed = sorted(root.rglob(bundle_name))\n    if len(packed) > 1:\n        raise ValueError("Ambiguous input bundle")\n    if packed:\n        archive = ZipFile(packed[0])\n        read = archive.read\n    else:\n        candidates = sorted(root.rglob("input_manifest.json"))\n        if len(candidates) != 1:\n            raise ValueError("Expected exactly one input manifest")\n        folder = candidates[0].parent\n        read = lambda name: (folder / name).read_bytes()\n        archive = None\n    try:\n        raw = read("input_manifest.json")\n        if digest_bytes(raw) != manifest_sha:\n            raise ValueError("Input manifest hash mismatch")\n        manifest = json.loads(raw)\n        locked_values = {}\n        for ref in manifest.get("threshold_locks", []):\n            payload = read(ref["member"])\n            if digest_bytes(payload) != ref["sha256"]:\n                raise ValueError("Threshold lock file hash mismatch")\n            lock = json.loads(payload)\n            body = {k: v for k, v in lock.items() if k != "locked_sha256"}\n            checks = [digest_bytes(json.dumps(body, sort_keys=True, ensure_ascii=False, indent=2).encode()), canonical(body)]\n            if lock.get("locked_sha256") not in checks:\n                raise ValueError("Threshold self-seal mismatch")\n            for entry in lock["entries"].values():\n                locked_values[entry["arm"], entry.get("seed", 42), entry["dataset"]] = entry["tau_star"]\n        groups = []\n        for group in manifest["groups"]:\n            if group["split"] == "test" and locked_values.get((group["arm"], group["seed"], group["dataset"])) != group.get("tau"):\n                raise ValueError("Test threshold is not the frozen validation lock")\n            payload = read(group["member"])\n            if digest_bytes(payload) != group["sha256"]:\n                raise ValueError("Frozen prompt-file hash mismatch")\n            prompts = jsonl(payload)\n            if len(prompts) != group["count"] or len({str(r["id"]) for r in prompts}) != len(prompts):\n                raise ValueError("Prompt census/duplicates mismatch")\n            for r in prompts:\n                if (r["arm"], r["seed"], r["dataset"], r["split"]) != (group["arm"], group["seed"], group["dataset"], group["split"]):\n                    raise ValueError("Prompt group identity mismatch")\n                if digest_bytes(r["prompt"].encode()) != r["prompt_sha256"]:\n                    raise ValueError("Individual prompt text hash mismatch")\n            groups.append((group, prompts))\n        legacy_raw = read("legacy_records.jsonl")\n        if digest_bytes(legacy_raw) != manifest["legacy_records_sha256"]:\n            raise ValueError("Legacy migration payload mismatch")\n        legacy = jsonl(legacy_raw)\n        expected = {(g["arm"], g["seed"], g["dataset"], g["split"], str(r["id"])): r for g, rs in groups for r in rs}\n        by_key = {}\n        for r in legacy:\n            key = (r["arm"], r["seed"], r["dataset"], r["split"], str(r["id"]))\n            if key in by_key or key not in expected:\n                raise ValueError("Legacy record ID is duplicate or absent from current inputs")\n            p = expected[key]\n            if (r["prompt_sha256"], r["label"], r["model_sha256"]) != (p["prompt_sha256"], p["label"], manifest["model_sha256"]):\n                raise ValueError("Legacy record prompt/label/model differs")\n            if not r.get("valid") or not math.isfinite(r["s"]) or not 0 <= r["s"] <= 1:\n                raise ValueError("Invalid legacy score")\n            by_key[key] = r\n        if len(by_key) != manifest["legacy_count"]:\n            raise ValueError("Legacy migration count mismatch")\n        return manifest, groups, by_key\n    finally:\n        if archive:\n            archive.close()\n\n\ndef key_for(g, row):\n    return g["arm"], g["seed"], g["dataset"], g["split"], str(row["id"])\n\n\ndef validate_saved(row, prompt, group, manifest, legacy, contract_sha, root=None):\n    key = key_for(group, row)\n    if (row.get("arm"), row.get("seed"), row.get("dataset"), row.get("split")) != key[:4]:\n        raise ValueError("Saved row identity mismatch")\n    if (row.get("prompt_sha256"), row.get("label"), row.get("model_sha256")) != (prompt["prompt_sha256"], prompt["label"], manifest["model_sha256"]):\n        raise ValueError("Saved row input identity mismatch")\n    if key in legacy:\n        if canonical(row) != canonical(legacy[key]):\n            raise ValueError("Historical record changed during migration")\n    elif row.get("runtime_contract_sha256") != contract_sha or row.get("inference_seed") != 4294967295 or row.get("record_origin") != "fresh_locked":\n        raise ValueError("Non-legacy row lacks runtime-lock evidence")\n    elif root is not None:\n        evidence = (Path(root) / row[\'runtime_provenance\']).resolve()\n        if not evidence.is_relative_to(Path(root).resolve()):\n            raise ValueError(\'Unsafe provenance path\')\n        payload = evidence.read_bytes()\n        if digest_bytes(payload) != row.get(\'runtime_provenance_sha256\'):\n            raise ValueError(\'Saved provenance hash mismatch\')\n        info = json.loads(payload)\n        if (info[\'runtime_contract_sha256\'], info[\'input_manifest_sha256\'], info[\'runner_sha256\']) != (contract_sha, manifest[\'_manifest_sha256\'], manifest[\'worker_sha256\']):\n            raise ValueError(\'Saved provenance identity mismatch\')\n        for filename, field in [(\'model_load_offload.log\',\'offload_log_sha256\'),(\'backend_build.log\',\'backend_build_log_sha256\')]:\n            if digest_bytes((evidence.parent / filename).read_bytes()) != info[field]:\n                raise ValueError(\'Saved native/build evidence hash mismatch\')\n    if not row.get("valid") or not math.isfinite(row["s"]) or not 0 <= row["s"] <= 1:\n        raise ValueError("Invalid saved score")\n\n\ndef main(args):\n    from grace_runtime_contract import load_contract, assert_runtime, provenance, load_locked_model, digest, command, EXPECTED_CONTRACT_SHA256\n    from transformers import AutoTokenizer\n    runtime = load_contract()\n    assert_runtime(runtime)\n    manifest, groups, legacy = load_inputs(args.input_root, args.bundle, args.manifest_sha)\n    manifest[\'_manifest_sha256\'] = args.manifest_sha\n    if digest(__file__) != manifest[\'worker_sha256\']:\n        raise ValueError(\'Worker source differs from pinned input manifest\')\n    if manifest["runtime_contract_sha256"] != EXPECTED_CONTRACT_SHA256:\n        raise ValueError("Input is for a different runtime contract")\n    root = Path(args.out)\n    root.mkdir(parents=True, exist_ok=True)\n    identity = {"input_manifest_sha256": args.manifest_sha, "runtime_contract_sha256": EXPECTED_CONTRACT_SHA256,\n                "model_sha256": manifest["model_sha256"], "constructor": runtime["constructor"],\n                "legacy_records_sha256": manifest["legacy_records_sha256"], "legacy_count": len(legacy),\n                "runtime_lock_scope": "newly inferred records only; imported historical records are byte/value preserved", "method": manifest["method"]}\n    previous = root / "run_identity.json"\n    if previous.exists() and json.loads(previous.read_text(encoding="utf-8")) != identity:\n        raise ValueError("Refusing checkpoint from a different input/runtime/migration identity")\n    atomic(previous, identity)\n    Path(root / "inference_runtime_lock.json").write_bytes(Path("/kaggle/working/inference_runtime_lock.json").read_bytes())\n    atomic(root / "legacy_migration.json", {"classification": manifest["legacy_classification"], "records": [{"key": list(k), "sha256": canonical(r)} for k, r in legacy.items()]})\n    saved = {}\n    for group, prompts in groups:\n        name = f"{group[\'arm\']}/seed_{group[\'seed\']}/{group[\'dataset\']}/{group[\'split\']}"\n        target = root / name / "records.jsonl"\n        target.parent.mkdir(parents=True, exist_ok=True)\n        existing = jsonl(target.read_bytes()) if target.exists() else []\n        lookup = {str(p["id"]): p for p in prompts}\n        done = {}\n        for row in existing:\n            uid = str(row["id"])\n            if uid in done or uid not in lookup:\n                raise ValueError("Unexpected or duplicate saved ID")\n            validate_saved(row, lookup[uid], group, manifest, legacy, EXPECTED_CONTRACT_SHA256, root)\n            done[uid] = row\n        imported = [r for k, r in legacy.items() if k[:4] == (group["arm"], group["seed"], group["dataset"], group["split"]) and str(r["id"]) not in done]\n        if imported:\n            with target.open("a", encoding="utf-8", newline="\\n") as f:\n                for row in imported:\n                    f.write(json.dumps(row, ensure_ascii=False, allow_nan=False) + "\\n")\n                    done[str(row["id"])] = row\n                f.flush(); os.fsync(f.fileno())\n        saved[name] = done\n    status = {"state": "setup", "groups": {name: len(rs) for name, rs in saved.items()}, "new_records_this_session": 0, "legacy_count": len(legacy)}\n    checkpoint = Path(args.checkpoint)\n    last_export = 0.0\n\n    def export(force=False):\n        nonlocal last_export\n        atomic(root / "status.json", status)\n        if force or time.monotonic() - last_export > 60:\n            temp = checkpoint.with_suffix(".zip.tmp")\n            with ZipFile(temp, "w", compression=ZIP_DEFLATED) as archive:\n                for path in sorted(root.rglob("*")):\n                    if path.is_file() and not path.name.endswith(".tmp"):\n                        archive.write(path, path.relative_to(root).as_posix())\n            os.replace(temp, checkpoint)\n            last_export = time.monotonic()\n\n    export(True)\n    stop_at = args.deadline - 300\n    try:\n        if all(len(saved[f"{g[\'arm\']}/seed_{g[\'seed\']}/{g[\'dataset\']}/{g[\'split\']}"]) == len(rs) for g, rs in groups):\n            status["state"] = "complete"\n            return\n        if time.time() >= stop_at:\n            status["state"] = "timed_out_before_model"\n            return\n        model_path = Path(args.model)\n        model_sha = digest(model_path)\n        if model_sha != runtime["model"]["sha256"] or model_sha != manifest["model_sha256"]:\n            raise ValueError("Model hash mismatch")\n        tokenizer = AutoTokenizer.from_pretrained(runtime["tokenizer"]["repo"], revision=runtime["tokenizer"]["revision"])\n        session = Path(tempfile.mkdtemp(prefix="locked_", dir=root))\n        provenance_file = session / "runtime_provenance.json"\n        info = provenance(runtime, __file__, provenance_file)\n        info["runtime_helper_sha256"] = digest(Path("/kaggle/working/grace_runtime_contract.py"))\n        info["input_manifest_sha256"] = args.manifest_sha\n        info["model_sha256"] = model_sha\n        build_log = session / \'backend_build.log\'\n        shutil.copyfile(\'/kaggle/working/grace_diagnostics/install_llama_cuda.log\', build_log)\n        info["backend_build_log_sha256"] = digest(build_log)\n        load_log = session / "model_load_offload.log"\n        tm = {"tokenizer": runtime["tokenizer"]["repo"], "tokenizer_revision": runtime["tokenizer"]["revision"], "prompt_budget_tokens": 8192}\n        llm = load_locked_model(runtime, model_path, tm, -1, load_log, model_sha)\n        info["offload_log_sha256"] = digest(load_log)\n        info["gpu_after_load"] = command(["nvidia-smi", "--query-gpu=index,uuid,name,memory.free,memory.total,driver_version", "--format=csv,noheader"])\n        atomic(provenance_file, info)\n        provenance_sha = digest(provenance_file)\n        label_ids = []\n        for label in ["0", "1"]:\n            hf = tokenizer.encode(label, add_special_tokens=False)\n            gguf = llm.tokenize(label.encode(), add_bos=False, special=False)\n            if len(hf) != 1 or hf != gguf:\n                raise ValueError("Binary token identity mismatch")\n            label_ids.append(hf[0])\n        status["state"] = "running"\n        for group, prompts in groups:\n            name = f"{group[\'arm\']}/seed_{group[\'seed\']}/{group[\'dataset\']}/{group[\'split\']}"\n            done = saved[name]\n            atomic(root / name / "group_identity.json", {**group, "runtime_lock_scope": identity["runtime_lock_scope"]})\n            with (root / name / "records.jsonl").open("a", encoding="utf-8", newline="\\n") as output:\n                for item in prompts:\n                    uid = str(item["id"])\n                    if uid in done:\n                        continue\n                    if time.time() >= stop_at:\n                        status["state"] = "timed_out"\n                        return\n                    messages = [{"role": "user", "content": item["prompt"]}]\n                    chat = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)\n                    hf_ids = tokenizer.apply_chat_template(messages, tokenize=True, add_generation_prompt=True, return_dict=False)\n                    ids = llm.tokenize(chat.encode(), add_bos=False, special=True)\n                    if ids != list(hf_ids) or len(ids) + 64 > 8192:\n                        raise ValueError(f"Tokenizer/context check failed at {uid}")\n                    start = time.monotonic()\n                    llm.reset()\n                    llm.eval(ids)\n                    score = binary_scores(llm.scores[llm.n_tokens - 1], label_ids)\n                    raw = llm.detokenize([score["greedy_token_id"]]).decode("utf-8", errors="replace").strip()\n                    row = {k: item[k] for k in ["id", "dataset", "split", "label", "prompt_sha256", "demo_ids", "tokens", "graph_missing"]}\n                    row.update(score, arm=group["arm"], seed=group["seed"], raw=raw, response_valid=raw in ("0", "1"), valid=True,\n                               model_sha256=model_sha, tokenizer_revision=runtime["tokenizer"]["revision"], token_ids={"0": label_ids[0], "1": label_ids[1]},\n                               runtime_contract_sha256=EXPECTED_CONTRACT_SHA256, inference_seed=runtime["constructor"]["seed"],\n                               runtime_provenance=provenance_file.relative_to(root).as_posix(), runtime_provenance_sha256=provenance_sha, record_origin="fresh_locked",\n                               elapsed_s=time.monotonic() - start, timestamp=datetime.now(timezone.utc).isoformat())\n                    if group["split"] == "test":\n                        row.update(tau=group["tau"], pred_tau=int(row["s"] >= group["tau"]))\n                    output.write(json.dumps(row, ensure_ascii=False, allow_nan=False) + "\\n")\n                    output.flush(); os.fsync(output.fileno())\n                    done[uid] = row\n                    status["groups"][name] = len(done)\n                    status["new_records_this_session"] += 1\n                    export()\n                    print(f"PROGRESS {name}: {len(done)}/{len(prompts)} id={uid}", flush=True)\n        status["state"] = "complete"\n    except BaseException as exc:\n        status.update(state="error", error_type=type(exc).__name__, error=str(exc))\n        raise\n    finally:\n        status["updated_at_utc"] = datetime.now(timezone.utc).isoformat()\n        export(True)\n\n\nif __name__ == "__main__":\n    parser = argparse.ArgumentParser()\n    for name in ["input-root", "bundle", "manifest-sha", "out", "checkpoint", "model"]:\n        parser.add_argument("--" + name, required=True)\n    parser.add_argument("--deadline", type=float, required=True)\n    main(parser.parse_args())\n', encoding='utf-8')


In [ ]:
deadline = SESSION_STARTED + MAX_RUNTIME_HOURS * 3600
try:
    run_logged([sys.executable, '/kaggle/working/kaggle_inference.py',
        '--input-root', str(KAGGLE_INPUT_ROOT), '--bundle', BUNDLE_NAME,
        '--manifest-sha', MANIFEST_SHA, '--out', str(OUT_ROOT),
        '--checkpoint', '/kaggle/working/' + CHECKPOINT_NAME,
        '--model', model_path, '--deadline', str(deadline)], 'locked_inference')
finally:
    diagnostics_zip()
